In [10]:
import pandas as pd
import pint
import pint_xarray
import xarray as xr

from util import create_region_graph, convert_to_xarray, aggregate_regions, extract_population
from constants import target_regions, region_mapping

In [11]:
ureg = pint.UnitRegistry()
ureg.define('terakilometer = 1e12 * kilometer')
ureg.define('milkilometer = 1e6 * kilometer')

pkmGlobal_original = pd.read_csv("image_3_4_data/pkmGlobalTot.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0)
tkmGlobal_original = pd.read_csv("image_3_4_data/tkmGlobalTot.csv", delimiter=";", usecols=range(8),skiprows=3).drop(0)
load_original = pd.read_csv("image_3_4_data/load.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0) #

In [12]:
pkmGlobal = convert_to_xarray(pkmGlobal_original, value_name = "pkm", variable = "mode")
tkmGlobal = convert_to_xarray(tkmGlobal_original, value_name = "tkm", variable = "mode")
load = convert_to_xarray(load_original, value_name = "load", variable = "mode")


In [13]:
# Convert units
pkmGlobal = pkmGlobal.pint.quantify('terakilometer') 
pkmGlobal= pkmGlobal.pint.to("km")

In [14]:
# Convert the region names in xr_image_output to match the target regions
knowledge_graph = create_region_graph()
xr_regions = pkmGlobal.coords["region"].values  # Extract current region names

xr_region_filter = knowledge_graph.find_relations_inverse(xr_regions, target_regions)



In [15]:
pkmGlobal_target_regions = aggregate_regions(pkmGlobal, xr_region_filter) #km
tkmGlobal_target_regions = aggregate_regions(tkmGlobal, xr_region_filter) # million km
load_target_regions = aggregate_regions(load, xr_region_filter, aggregation = 'mean') #person/vehicle

load_target_regions

<xarray.DataArray 'load' (region: 13, year: 2, mode: 7)> Size: 1kB
array([[[1.091914  , 0.9804317 , 1.        , 1.        , 1.        ,
         1.091914  , 1.        ],
        [0.9681365 , 0.9287609 , 1.        , 1.        , 1.        ,
         0.9681365 , 1.        ]],

       [[1.255597  , 1.044035  , 1.        , 1.        , 1.        ,
         1.255597  , 1.        ],
        [1.071097  , 0.9719759 , 1.        , 1.        , 1.        ,
         1.071097  , 1.        ]],

       [[1.345928  , 1.07719   , 1.        , 1.        , 1.        ,
         1.345928  , 1.        ],
        [1.080504  , 0.9758081 , 1.        , 1.        , 1.        ,
         1.080504  , 1.        ]],

       [[1.615049  , 1.169273  , 1.        , 1.        , 1.        ,
         1.615049  , 1.        ],
        [1.20791   , 1.026001  , 1.        , 1.        , 1.        ,
         1.20791   , 1.        ]],

...

       [[1.385803  , 1.090993  , 1.        , 1.        , 1.        ,
         1.385803  , 1.        ],
        [1.16456125, 1.00889295, 1.        , 1.        , 1.        ,
         1.16456125, 1.        ]],

       [[1.78320625, 1.2200975 , 1.        , 1.        , 1.        ,
         1.78320625, 1.        ],
        [1.44547725, 1.11073725, 1.        , 1.        , 1.        ,
         1.44547725, 1.        ]],

       [[1.05826   , 0.9667165 , 1.        , 1.        , 1.        ,
         1.05826   , 1.        ],
        [0.9748507 , 0.9316539 , 1.        , 1.        , 1.        ,
         0.9748507 , 1.        ]],

       [[1.131967  , 0.9964547 , 1.        , 1.        , 1.        ,
         1.131967  , 1.        ],
        [1.022967  , 0.9520729 , 1.        , 1.        , 1.        ,
         1.022967  , 1.        ]]])
Coordinates:
  * year     (year) int64 16B 2019 2060
  * mode     (mode) object 56B 'bus' 'car' 'cycling' ... 'train' 'walking'
  * region   (region) object 104B 'Canada' 'Central Europe' ... 'Western Europe'

In [16]:
pop_million = extract_population() # million
pop = pop_million * 1e6 
pop = pop.drop_vars("variable").squeeze()

tkmGlobal_target_regions = tkmGlobal_target_regions * 1e6 #from million tkm to tkm


In [17]:
pkm_per_capita = pkmGlobal_target_regions / pop
tkm_per_capita = tkmGlobal_target_regions / pop

In [20]:
load_df

,region,year,mode,load
0,Canada,2019,bus,1.091914
1,Canada,2019,car,0.980432
2,Canada,2019,cycling,1.000000
3,Canada,2019,highspeed train,1.000000
4,Canada,2019,plane,1.000000
...,...,...,...,...
177,Western Europe,2060,cycling,1.000000
178,Western Europe,2060,highspeed train,1.000000
179,Western Europe,2060,plane,1.000000
180,Western Europe,2060,train,1.022967


In [ ]:
# Assign names to the DataArrays
pkm_per_capita.name = 'pkm_per_capita'
tkm_per_capita.name = 'tkm_per_capita'
load_target_regions.name = 'load'

# Step 1: Convert xarrays to DataFrames
pkm_per_capita_df = pkm_per_capita.to_dataframe().reset_index()
tkm_per_capita_df = tkm_per_capita.to_dataframe().reset_index()
load_df = load_target_regions.to_dataframe().reset_index()


# Step 3: Merge the DataFrames into a single DataFrame
# Assuming all DataFrames have the same dimensions and matching columns, merge them on the common indices
#merged_df = pd.merge(pkm_per_capita_df, tkm_per_capita_df, on=["region", "year", "mode"])
#merged_df = pd.merge(merged_df, load_df, on=["region", "year", "mode"])


# Step 4: Export to CSV
#merged_df.to_csv("output_data.csv", index=False)


c:\Users\5982758\AppData\Local\anaconda3\envs\materials\lib\site-packages\xarray\core\variable.py:338: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


In [30]:
# Melt the pkm_per_capita DataFrame
pkm_per_capita_df_melted = pkm_per_capita_df.melt(id_vars=["region", "year", "mode"], 
                                                   value_vars=["pkm_per_capita"], 
                                                   var_name="variable", value_name="value")

# Assign the variable name for pkm_per_capita
pkm_per_capita_df_melted["variable"] = "pkm_per_capita"

# Melt the tkm_per_capita DataFrame
tkm_per_capita_df_melted = tkm_per_capita_df.melt(id_vars=["region", "year", "mode"], 
                                                   value_vars=["tkm_per_capita"], 
                                                   var_name="variable", value_name="value")

# Assign the variable name for tkm_per_capita
tkm_per_capita_df_melted["variable"] = "tkm_per_capita"

# Merge the two melted DataFrames
merged_df = pd.merge(pkm_per_capita_df_melted, tkm_per_capita_df_melted, 
                     on=["region", "year", "mode",'variable','value'], how="outer")

load_df_melted = load_df.melt(id_vars=["region", "year", "mode"], 
                              value_vars=["load"], 
                              var_name="variable", value_name="value")

# Assign the variable name for load
load_df_melted["variable"] = "load"

# Merge the two melted DataFrames
merged_df = pd.merge(merged_df, load_df_melted, 
                     on=["region", "year", "mode",'variable','value'], how="outer")

# Optionally, you may want to combine or rearrange columns as needed
#merged_df = merged_df[['region', 'year', 'mode', 'variable', 'value']]

merged_df.to_csv("output_data.csv", index=False)


In [31]:
merged_df

,region,year,mode,variable,value
0,Canada,2019,bus,load,1.091914
1,Canada,2019,bus,pkm_per_capita,2882.700412
2,Canada,2019,car,load,0.980432
3,Canada,2019,car,pkm_per_capita,28447.022858
4,Canada,2019,cycling,load,1.0
...,...,...,...,...,...
515,Western Europe,2060,train,load,1.022967
516,Western Europe,2060,train,pkm_per_capita,1809.532643
517,Western Europe,2060,train,tkm_per_capita,351.616967
518,Western Europe,2060,walking,load,1.0
